# QUIZ - 2025-08

In [5]:
import os
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

In [6]:
from shop.models import *
from django.db.models import *
from django.db.models.functions import *
from datetime import datetime, timedelta, time

## PART 1 - CRUD Operations (2 คะแนน)

1.1 สร้างข้อมูลนัดหมาย (`Appointment`) ของลูกค้า "Customer 4" ซึ่งต้องการจองนัดหมายบริการชื่อ "Massage" ที่ใช้เวลา (`duration`) น้อยที่สุด 

โดยสร้างนัดหมายในวันพรุ่งนี้ เวลา 13:00 น. 

(0.5 คะแนน)

*หมายเหตุ 1: จะต้อง get ข้อมูลมาโดยใช้การ query ด้วยชื่อตามที่โจทย์ว่าเท่านั้น ห้ามใช้ id ใน database*

*หมายเหตุ 2: ต้องใช้ `datetime.timedelta` ในการหาวันพรุ่งนี้*

In [10]:
print(datetime.now().date())
print(datetime.now().time())
print(datetime.now())

2025-08-28
08:03:57.414485
2025-08-28 08:03:57.414485


In [46]:
service = Service.objects.filter(name="Massage").aggregate(Min("duration"))
print(service)

{'duration__min': datetime.timedelta(seconds=2700)}


In [55]:
# CODE HERE
customer4_cus = Customer.objects.get(name="Customer 4")
customer4_ser = Service.objects.filter(name="Massage",
                                       duration=service["duration__min"]).first()
tomorrow = datetime.now().date()+timedelta(days=1)
customer4 = Appointment.objects.create(customer=customer4_cus, 
                                       service=customer4_ser,
                                       appointment_date=tomorrow, 
                                      appointment_time=time(13, 0))

In [57]:
# Check result
print("Appoint ID: %s, Appoint Date: %s, Appoint Time: %s"%(
    customer4.id, customer4.appointment_date, customer4.appointment_time
))
print("Service ID: %s, Service Name: %s, Provider Name: %s, Customer Name: %s"%(
    customer4.service.id, customer4.service.name, customer4.service.service_provider.name, customer4.customer.name
))

Appoint ID: 16, Appoint Date: 2025-08-29, Appoint Time: 13:00:00
Service ID: 9, Service Name: Massage, Provider Name: Provider 3, Customer Name: Customer 4


1.2 ทำการแก้ไขนัดหมาย ณ วันที่ 2024-08-17 ของ "Customer 2" โดยเปลี่ยนจากเดิมที่นัดมาให้บริการ "Haircut" ของ "Provider 4" เป็น "Manicure" ของ "Provider 2" แทน

(0.25 คะแนน)

*หมายเหตุ: จะต้อง get ข้อมูลมาโดยใช้การ query ด้วยชื่อตามที่โจทย์ว่าเท่านั้น ห้ามใช้ id ใน database*

In [72]:
# CODE HERE
cus2 = Customer.objects.get(name="Customer 2")
appoint2 = Appointment.objects.get(customer=cus2, appointment_date="2024-08-17")
newser = Service.objects.get(name="Manicure", service_provider__name="Provider 2")
appoint2.service = newser #เปลี่ยน foreignkey
appoint2.save()

In [73]:
# Check result
print("Appoint ID: %s, Appoint Date: %s, Appoint Time: %s" % (
    appoint2.id, appoint2.appointment_date, appoint2.appointment_time
))
print("Service ID: %s, Service Name: %s, Provider Name: %s, Customer Name: %s" % (
    appoint2.service.id, appoint2.service.name, appoint2.service.service_provider.name, appoint2.customer.name
))

Appoint ID: 10, Appoint Date: 2024-08-17, Appoint Time: 14:00:00
Service ID: 5, Service Name: Manicure, Provider Name: Provider 2, Customer Name: Customer 2


1.3 ทำตามขั้นตอนดังนี้

1. สร้างข้อมูลลูกค้า (`Customer`) ใหม่ที่มีชื่อ, อีเมล และเบอร์โทร เป็นของนักศึกษา _ตัวอย่าง (65070xxx, 65070xxx@kmitl.ac.th, 000-000-0000)_
2. สร้างข้อมูลหมวดหมู่บริการ (`ServiceCategory`) ใหม่ที่มีชื่อ "Energy & Body Care" และมีรายละเอียดว่า "Service related to body & mind"
3. สร้างข้อมูลบริการ (`Service`) ที่มีข้อมูลดังนี้

    - Name: 'Force training'
    - Description: 'Enhance your power and control. Involve learning various techniques, such as Force push, telekinesis'
    - Duration: '12:00:00'
    - Price: 3500.66
    - บริการโดย "Provider 1" และ "Provider 4" 
    (Hint: ดังนั้นต้องสร้างบริการ (`Service`) 2 รายการ ของแต่ละผู้ให้บริการ (`ServiceProvider`))

4. เพิ่มบริการที่สร้างเข้าไปในหมวดหมู่บริการ (`ServiceCategory`) "Energy & Body Care"
5. สร้างข้อมูลนัดหมาย (`Appointment`) ของลูกค้าที่พึ่งสร้างมาใหม่ในข้อ 1 ซึ่งต้องการรับบริการ "Force training" ของ "Provider 4" ณ วันนี้ และเวลาปัจจุบัน (Hint: `datetime.now()`)
6. ทำการย้ายบริการ (`Service`) ในหมวดหมู่บริการ (`ServiceCategory`) "Body Care" ทั้งหมดไปยังหมวดหมู่ "Energy & Body Care"
7. ทำการลบหมวดหมู่บริการ (`ServiceCategory`) "Body Care"

(1 คะแนน)

*หมายเหตุ 1: จะต้อง get ข้อมูลมาโดยใช้การ query ด้วยชื่อตามที่โจทย์ว่าเท่านั้น ห้ามใช้ id ใน database*

*หมายเหตุ 2: เบอร์โทรไม่จำเป็นต้องเป็นของจริง*

In [91]:
# CODE HERE
newcus = Customer.objects.create(name="Indira", email="indira@gmail.com", phone="0987654321")
pro1 = ServiceProvider.objects.get(name="Provider 1")
pro4 = ServiceProvider.objects.get(name="Provider 4")
newser1 = Service.objects.create(service_provider=pro1, name="Force training", description="Enhance your power and control. Involve learning various techniques, such as Force push, telekinesis", duration=time(12, 0), price=3500.66)
newser4 = Service.objects.create(service_provider=pro4, name="Force training", description="Enhance your power and control. Involve learning various techniques, such as Force push, telekinesis", duration=time(12, 0), price=3500.66)
new_cat = ServiceCategory.objects.create(name="Energy & Body Care", description="Service related to body & mind")
new_cat.services.add(newser1, newser4)
appoint3 = Appointment.objects.create(customer=newcus, 
                                       service=newser4,
                                       appointment_date=datetime.now().date(), 
                                      appointment_time=datetime.now().replace(microsecond=0).time())
body = Service.objects.filter(categories__name="Body Care")
new_cat.services.add(*body)
oldcat = ServiceCategory.objects.get(name="Body Care")
oldcat.delete()

(6, {'shop.ServiceCategory_services': 5, 'shop.ServiceCategory': 1})

In [92]:
# Check result
print("Number of services in Energy & Body Care category: %d" % new_cat.services.count())
print("Appoint Date: %s, Appoint Time: %s"%(appoint3.appointment_date, appoint3.appointment_time))
print("Appointment's Service: %s, Provider: %s, Customer: %s" % (
    appoint3.service.name, appoint3.service.service_provider.name, appoint3.customer.name
))

Number of services in Energy & Body Care category: 7
Appoint Date: 2025-08-28, Appoint Time: 09:20:45.685110
Appointment's Service: Force training, Provider: Provider 4, Customer: Indira


1.4 ทำการลบบริการที่ไม่เคยมีใครจองเลย (0.25 คะแนน)

*หมายเหตุ: จะต้อง get ข้อมูลมาโดยใช้การ query ด้วยชื่อตามที่โจทย์ว่าเท่านั้น ห้ามใช้ id ใน database*

In [100]:
before_delete = Service.objects.annotate(no=Count("appointments")).filter(no=0).count()

In [101]:
# แสดงจำนวนบริการที่ไม่มีการนัดหมายเลย ก่อนลบ

print(f"Number of service with no appointment (before): {before_delete}")

Number of service with no appointment (before): 12


In [103]:
# ทำการลบ
Service.objects.annotate(no=Count("appointments")).filter(no=0).delete()
after_delete = Service.objects.annotate(no=Count("appointments")).filter(no=0).count()

In [104]:
# แสดงจำนวนบริการที่ไม่มีการนัดหมายเลย หลังลบ

print(f"Number of service with no appointment (after): {after_delete}")

Number of service with no appointment (after): 0


## PART 2 - Making Queries (3 คะแนน)

สำหรับ PART 2 ให้ทำการ reset DB และ import ข้อมูลใน `service.sql` เข้าไปใหม่

2.1 ให้แสดงข้อมูลบริการ (`Service`) ที่มีราคาอยู่ในช่วง 25 ถึง 50 บาท

(0.5 คะแนน)

**Expected Output**

*แสดงผลเรียงตาม `id` จากน้อยไปมาก*

```
ID: 2, NAME: Manicure, PRICE: 30
ID: 5, NAME: Manicure, PRICE: 30
ID: 8, NAME: Manicure, PRICE: 30
ID: 11, NAME: Manicure, PRICE: 30
```

In [7]:
# CODE HERE
result = Service.objects.filter(price__lte=50, price__gte=25).order_by('id')

In [10]:
# Print results
for i in result:
    print(f"ID: {i.id}, NAME: {i.name}, PRICE: {i.price}")

ID: 2, NAME: Manicure, PRICE: 30.00
ID: 5, NAME: Manicure, PRICE: 30.00
ID: 8, NAME: Manicure, PRICE: 30.00


2.2 แสดงรายชื่อลูกค้า (`Customer`) ทุกคนที่มีการนัดหมายเข้าใช้บริการจากผู้ให้บริการ (`ServiceProvider`) ชื่อ "Provider 3"

(0.5 คะแนน)

**Expected Output**

*แสดงผลเรียงตาม `id` จากน้อยไปมาก*

```
ID: 1, NAME: Customer 1
ID: 4, NAME: Customer 4
```

In [15]:
# CODE HERE
result2 = Customer.objects.filter(appointments__service__service_provider__name="Provider 3").order_by("id")

In [16]:
# Print results
for i in result2:
    print(f"ID: {i.id}, NAME: {i.name}")

ID: 1, NAME: Customer 1
ID: 4, NAME: Customer 4
ID: 4, NAME: Customer 4


2.3 ให้แสดงข้อมูลประเภทบริการ (`ServiceCategory`) ทั้งหมด และแสดงว่าแต่ละประเภทมีกี่บริการ (`Service`) ที่อยู่ในประเภทนั้น ๆ

*หมายเหตุ: ห้ามใช้ `filter()` และควรใช้ `annotate()`*

(0.5 คะแนน)

**Expected Output**

*แสดงผลเรียงจากจำนวน SERVICE COUNT มากไปน้อย*

```
ID: 3, NAME: Body Care, SERVICE COUNT: 5
ID: 2, NAME: Nail Care, SERVICE COUNT: 4
ID: 1, NAME: Hair Care, SERVICE COUNT: 4
```

In [29]:
# CODE HERE
result3 = ServiceCategory.objects.annotate(service_count=Count("services")).order_by("-service_count")

In [30]:
# Print results
for i in result3:
    print(f"ID: {i.id}, NAME: {i.name}, SERVICE COUNT: {i.service_count}")

ID: 9, NAME: Energy & Body Care, SERVICE COUNT: 5
ID: 2, NAME: Nail Care, SERVICE COUNT: 3
ID: 1, NAME: Hair Care, SERVICE COUNT: 2
ID: 7, NAME: Energy & Body Care, SERVICE COUNT: 1
ID: 6, NAME: Energy & Body Care, SERVICE COUNT: 1
ID: 8, NAME: Energy & Body Care, SERVICE COUNT: 1
ID: 4, NAME: Energy & Body Care, SERVICE COUNT: 0
ID: 5, NAME: Energy & Body Care, SERVICE COUNT: 0


2.4 ให้แสดงรายชื่อลูกค้า (`Customer`) ที่มีนัดหมายล่าสุดเป็นวันหลังจากวันที่ 14 สิงหาคม 2024 

หมายเหตุ: ต้องมีการใช้ `Subquery()` อย่างน้อย 1 จุด

(0.5 คะแนน)

**Expected Output**
```
ID: 1, NAME: Customer 1, LATEST APPOINTMENT: 2024-08-15
ID: 2, NAME: Customer 2, LATEST APPOINTMENT: 2024-08-17
```

In [32]:
# CODE HERE
latest = Appointment.objects.filter(customer=OuterRef("pk")).order_by("-appointment_date")
cussss = Customer.objects.annotate(each=Subquery(
    latest.values("appointment_date")[:1])).filter(each__gt='2024-08-14')

In [36]:
# Print results
for i in cussss:
    print(f"ID: {i.id}, NAME: {i.name}, LATEST APPOINTMENT: {i.each}")

ID: 1, NAME: Customer 1, LATEST APPOINTMENT: 2024-08-15
ID: 2, NAME: Customer 2, LATEST APPOINTMENT: 2024-08-17
ID: 4, NAME: Customer 4, LATEST APPOINTMENT: 2025-08-29
ID: 10, NAME: Indira, LATEST APPOINTMENT: 2025-08-28
ID: 11, NAME: Indira, LATEST APPOINTMENT: 2025-08-28
ID: 12, NAME: Indira, LATEST APPOINTMENT: 2025-08-28
ID: 13, NAME: Indira, LATEST APPOINTMENT: 2025-08-28


2.5 ให้รวมการใช้จ่ายของลูกค้าทุกคน และแสดงผล (1 คะแนน)

*hint: ดูจาก `Appointment` ทั้งหมดของลูกค้า และรวม `Service.price` ทั้งหมด*

**Expected Output**
```
ID: 1, NAME: Customer 1, TOTAL SPEND: 210
ID: 2, NAME: Customer 2, TOTAL SPEND: 180
ID: 3, NAME: Customer 3, TOTAL SPEND: 230
ID: 4, NAME: Customer 4, TOTAL SPEND: 150
```

In [44]:
# CODE HERE
result4 = Customer.objects.annotate(total=Sum("appointments__service__price")).order_by("id")

In [42]:
# Print results
for i in result4:
    print(f"ID: {i.id}, NAME: {i.name}, TOTAL SPEND: {i.total}")

ID: 1, NAME: Customer 1, TOTAL SPEND: 210.00
ID: 2, NAME: Customer 2, TOTAL SPEND: 190.00
ID: 3, NAME: Customer 3, TOTAL SPEND: 230.00
ID: 4, NAME: Customer 4, TOTAL SPEND: 250.00
ID: 5, NAME: Indira, TOTAL SPEND: None
ID: 6, NAME: Indira, TOTAL SPEND: None
ID: 7, NAME: Indira, TOTAL SPEND: None
ID: 8, NAME: Indira, TOTAL SPEND: None
ID: 9, NAME: Indira, TOTAL SPEND: None
ID: 10, NAME: Indira, TOTAL SPEND: 3500.66
ID: 11, NAME: Indira, TOTAL SPEND: 3500.66
ID: 12, NAME: Indira, TOTAL SPEND: 3500.66
ID: 13, NAME: Indira, TOTAL SPEND: 3500.66


2.6 ให้แสดงรายชื่อผู้ให้บริการ (`ServiceProvider`) นับจำนวนนัดหมาย (`Appointment`) และยอดรวมยอดขาย (`Appointment.service.price`) ของแต่ละผู้ให้บริการ

*หมายเหตุ: ควรใช้ `annotate()`*

(1 คะแนน - ข้อนี้คะแนนแถมถึงทำได้ก็จะได้เต็ม 5 อยู่ดีนะครับ เผื่อข้ออื่นๆ ก่อนหน้ามีพลาดทำผิด)

**Expected Output**

*แสดงผลเรียงจากยอดรวมยอดขาย (REVENUE) มากไปน้อย*

```
ID: 1, NAME: Provider 1, APPOINT COUNT: 7, REVENUE: 410
ID: 2, NAME: Provider 2, APPOINT COUNT: 5, REVENUE: 210
ID: 3, NAME: Provider 3, APPOINT COUNT: 2, REVENUE: 130
ID: 4, NAME: Provider 4, APPOINT COUNT: 1, REVENUE: 20
```

In [56]:
# CODE HERE
apo = ServiceProvider.objects.annotate(coapo=Count("services__appointments"), 
                                       coapo2=Sum("services__appointments__service__price")).order_by("-coapo2")

In [57]:
# Print results
for i in apo:
    print(f"ID: {i.id}, NAME: {i.name}, APPOINT COUNT: {i.coapo}, REVENUE: {i.coapo2}")

ID: 4, NAME: Provider 4, APPOINT COUNT: 4, REVENUE: 14002.64
ID: 1, NAME: Provider 1, APPOINT COUNT: 7, REVENUE: 410.00
ID: 2, NAME: Provider 2, APPOINT COUNT: 6, REVENUE: 240.00
ID: 3, NAME: Provider 3, APPOINT COUNT: 3, REVENUE: 230.00


**โจทย์ 1 — Appointment ล่าสุดต่อแต่ละ Customer**

แสดงรายชื่อลูกค้า (Customer) พร้อม บริการล่าสุดที่จอง และวันที่นัดล่าสุด
`เงื่อนไข: ต้องใช้ Subquery`

In [82]:
las = Appointment.objects.filter(customer=OuterRef("pk")).order_by("-appointment_date")
cusnew = Customer.objects.annotate(
    laslas=Subquery(las.values("appointment_date")[:1]), 
    laslaslas=Subquery(las.values("service__name")[:1]))
for i in cusnew:
    print(f"NAME: {i.name}, SERVICE: {i.laslaslas}, DATE: {i.laslas}")

NAME: Customer 1, SERVICE: Massage, DATE: 2024-08-15
NAME: Customer 2, SERVICE: Manicure, DATE: 2024-08-17
NAME: Customer 3, SERVICE: Manicure, DATE: 2024-08-12
NAME: Customer 4, SERVICE: Massage, DATE: 2025-08-29
NAME: Indira, SERVICE: None, DATE: None
NAME: Indira, SERVICE: None, DATE: None
NAME: Indira, SERVICE: None, DATE: None
NAME: Indira, SERVICE: None, DATE: None
NAME: Indira, SERVICE: None, DATE: None
NAME: Indira, SERVICE: Force training, DATE: 2025-08-28
NAME: Indira, SERVICE: Force training, DATE: 2025-08-28
NAME: Indira, SERVICE: Force training, DATE: 2025-08-28
NAME: Indira, SERVICE: Force training, DATE: 2025-08-28


**โจทย์ 2 — ราคาสูงสุดต่อ Service Provider**

แสดงรายชื่อ `Service Provider` พร้อมราคาบริการสูงสุดของเขา
ใช้ `Subquery` เพื่อดึง `price` สูงสุดจากตาราง `Service`

In [89]:
exp = Service.objects.filter(service_provider=OuterRef("pk")).order_by("-price")
exp2 = ServiceProvider.objects.annotate(pri=Subquery(exp.values("price")[:1]))
for i in exp2:
    print(f"Name: {i.name}, Price: {i.pri}")

Name: Provider 1, Price: 100.00
Name: Provider 2, Price: 100.00
Name: Provider 3, Price: 100.00
Name: Provider 4, Price: 3500.66


**โจทย์ 3 — ลูกค้าที่จองบริการแพงที่สุด**

แสดง Customer ที่มีนัดหมายบริการราคาแพงที่สุดในแต่ละวัน
`ใช้ Subquery เพื่อหา MAX(price) ของวันนั้น ๆ`

In [90]:
# Subquery: ราคาสูงสุดของวันนั้น
max_price_sub = Appointment.objects.filter(
    appointment_date=OuterRef("appointment_date")
).order_by("-service__price")

# Annotate appointment แต่ละตัวด้วยราคาสูงสุดของวันนั้น
appoints = Appointment.objects.annotate(
    max_price_day=Subquery(max_price_sub.values("service__price")[:1])
).filter(service__price=F('max_price_day'))

# แสดงผล
for a in appoints:
    print(f"{a.appointment_date} - {a.customer.name} - {a.service.name} - Price: {a.service.price}")

2024-08-04 - Customer 1 - Haircut - Price: 20.00
2024-08-07 - Customer 3 - Massage - Price: 100.00
2024-08-10 - Customer 1 - Manicure - Price: 30.00
2024-08-11 - Customer 2 - Massage - Price: 100.00
2024-08-14 - Customer 4 - Manicure - Price: 30.00
2024-08-15 - Customer 1 - Massage - Price: 100.00
2024-08-08 - Customer 2 - Manicure - Price: 30.00
2024-08-05 - Customer 3 - Massage - Price: 100.00
2024-08-12 - Customer 4 - Massage - Price: 100.00
2024-08-10 - Customer 1 - Manicure - Price: 30.00
2025-08-29 - Customer 4 - Massage - Price: 100.00
2024-08-17 - Customer 2 - Manicure - Price: 30.00
2025-08-28 - Indira - Force training - Price: 3500.66
2025-08-28 - Indira - Force training - Price: 3500.66
2025-08-28 - Indira - Force training - Price: 3500.66
2025-08-28 - Indira - Force training - Price: 3500.66


**โจทย์ 4 — จำนวน appointment ล่าสุดต่อ Service**

แสดง Service แต่ละตัว พร้อมจำนวน appointment ล่าสุด ของมัน
`ใช้ Subquery + OuterRef + Count`

In [91]:
# Subquery: นับ appointment ของ service
appointment_sub = Appointment.objects.filter(
    service=OuterRef("pk")
).order_by("-appointment_date")  # สามารถ limit [:1] หากอยากได้แค่ล่าสุด

# Annotate Service ด้วยจำนวน appointment
services = Service.objects.annotate(
    last_appointments=Subquery(
        appointment_sub.values("service").annotate(c=Count("id")).values("c")[:1],
        output_field=IntegerField()
    )
)

# แสดงผล
for s in services:
    print(f"{s.name} - Latest Appointments: {s.last_appointments}")

Haircut - Latest Appointments: 1
Manicure - Latest Appointments: 1
Massage - Latest Appointments: 1
Haircut - Latest Appointments: 1
Manicure - Latest Appointments: 1
Massage - Latest Appointments: 1
Manicure - Latest Appointments: 1
Massage - Latest Appointments: 1
Force training - Latest Appointments: 1
Force training - Latest Appointments: 1
Force training - Latest Appointments: 1
Force training - Latest Appointments: 1
